# day-14-vector-databases — worked solutions & answer key

Solutions to the exercises in [`../lesson.ipynb`](../lesson.ipynb), plus the self-check answer key. **Try each exercise yourself first** — the value is in the attempt, not the answer.

In [8]:
# ---- Solution 1 ----
for nlist in [16, 64, 256]:
    iv = IVF(X, nlist=nlist)
    for nprobe in range(1, nlist+1):
        res = [iv.search(q, nprobe=nprobe) for q in Q]
        res = [np.pad(r,(0,10-len(r)),constant_values=-1) if len(r)<10 else r for r in res]
        rec = recall_at_10(res)
        if rec >= 0.95:
            scanned = np.mean([sum(len(iv.lists[c]) for c in np.argsort(-(iv.C@q))[:nprobe]) for q in Q])
            print(f"nlist={nlist:3d}: nprobe={nprobe:2d} -> recall {rec:.3f}, ~{scanned:.0f} vecs/query "
                  f"({100*scanned/N:.0f}% of corpus)")
            break

nlist= 16: nprobe= 2 -> recall 0.973, ~1026 vecs/query (13% of corpus)
nlist= 64: nprobe= 2 -> recall 0.990, ~263 vecs/query (3% of corpus)


nlist=256: nprobe= 5 -> recall 0.972, ~170 vecs/query (2% of corpus)


In [9]:
# ---- Solution 2 ----
print(f"{'M':>4} {'graph MB':>10} {'% of vectors':>13} {'recall@10 (ef=64)':>18}")
for M in [8, 16, 32]:
    g = NSW(X, M=M)
    gmb = sum(len(x) for x in g.graph) * 4 / 1e6
    r = np.array([g.search(q, ef=64) for q in Q])
    print(f"{M:>4} {gmb:>10.2f} {100*gmb/(X.nbytes/1e6):>12.0f}% {recall_at_10(r):>18.3f}")
print("S2: higher M -> more memory + build time, higher recall/reachability. Diminishing past M~32.")

   M   graph MB  % of vectors  recall@10 (ef=64)


   8       0.51           17%              0.984


  16       1.02           33%              0.993


  32       2.04           67%              0.998
S2: higher M -> more memory + build time, higher recall/reachability. Diminishing past M~32.


In [10]:
# ---- Solution 5 ----
s = 0.1; k = 10
for safety in [1.0, 1.5, 2.0, 3.0]:
    fetch = int(k / s * safety)
    survivors = []
    for qi in range(100):
        cand = nsw.search(Q[qi], k=fetch, ef=max(fetch, 64))
        survivors.append(sum(meta_tenant[c] == 3 for c in cand))
    print(f"safety {safety}: fetch {fetch:3d} -> mean survivors {np.mean(survivors):.1f} (need >= {k})")
print("S5: fetch = k/s covers the average; ~1.5-2x safety covers the variance.")

safety 1.0: fetch 100 -> mean survivors 9.8 (need >= 10)


safety 1.5: fetch 150 -> mean survivors 15.1 (need >= 10)


safety 2.0: fetch 200 -> mean survivors 20.1 (need >= 10)


safety 3.0: fetch 300 -> mean survivors 30.4 (need >= 10)
S5: fetch = k/s covers the average; ~1.5-2x safety covers the variance.


### Solutions 3, 4, 6 (worked)

**S3:** for each query, `len(set(kmeans_cell_of(gt_i) for gt_i in GT[q]))`. Queries whose GT
neighbours span 2+ cells are exactly the ones IVF-`nprobe=1` misses (it only searches one
cell); HNSW's graph walk crosses the boundary freely. This is IVF's structural weakness.

**S4:** as selectivity → 0, `post_filter` with a fixed `fetch=10` returns ~`10·s` results
(often 0–1) and recall collapses. It "breaks" once `s < ~0.3` for `fetch=k`; over-fetch pushes
that threshold down but never removes it — very selective filters need a filter-aware index.

**S6:** (a) **pgvector** — it's a Rails/Postgres app, 200k vectors is trivial, and the filters
are plain SQL; runner-up Qdrant. (b) **Pinecone** — 500M chunks and spiky traffic is the
serverless-managed sweet spot; runner-up Milvus (if self-hosting) or Turbopuffer. (c)
**Weaviate** — native BM25+vector hybrid in one query over 5M docs; runner-up Qdrant (hybrid
added recently) or pgvector with `tsvector` + manual RRF.

### Answer key
1. ~10⁵–10⁶. A brute-force `D @ q` matmul is memory-bandwidth-bound and extremely fast on
   modern hardware; below ~1M vectors an ANN index adds complexity, memory, and recall loss
   for little latency gain.
2. The number of k-means cells searched per query. `nprobe = nlist` searches every cell = exact
   flat search (recall 1.0, no speedup).
3. IVF's cells are defined by k-means centroids fit to the data; inserting many new vectors
   shifts the true cluster structure away from the frozen centroids, degrading recall. HNSW
   just adds a node and links it to its nearest neighbours — the graph adapts incrementally.
4. `M` = number of neighbour links per node (fixed at build; trades memory/build-time for
   recall). `ef` = search beam width (set per query; trades latency for recall).
5. The ANN retrieves the global nearest neighbours, then you discard those failing the filter.
   If the filter matches only a small fraction of the corpus, most retrieved neighbours are
   discarded and you return far fewer than `k` — or none — with poor recall.
6. pgvector: the data and filters already live in Postgres, 800k vectors is well within its
   comfort zone, and you get transactional consistency and SQL joins for free. Pinecone/
   Weaviate would mean syncing data into a second system and reimplementing the joins/filters
   you already have.
7. recall@k and p95 latency on your real data *at your real filter selectivity and scale*
   (plus cost at that scale).